# C2 · A single point moves the line: leverage versus residual

**Type**: Break the model · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Add the same vertical outlier in two different positions and measure how much it moves the slope: in the center of the
data almost nothing happens, at the edges the model breaks. Then compare robust estimators.

## Dataset

Synthetic, 50 points: `x` uniform in [0, 10], `y = 2 + 1.5·x + noise` (standard deviation 1). True slope: 1.5.

## Educational purpose

Separate two notions that get confused: a point is **anomalous** (large residual) and a point is **influential** (it moves the
model). To be influential you need high leverage, that is being far from the center of the data. You will then see two robust
estimators, Huber and RANSAC, which reduce the problem in different ways.

## Task

1. Write `slope(x, y, model=None)`: it trains the model (by default `LinearRegression`) with `x` as the only feature and returns the slope as a number. Compute `slope_base` on the clean data.
2. Build two contaminated datasets by adding a point with `y = -20`: one at `x = 10` (edge, `x_lev, y_lev`) and one at `x = 5` (center, `x_center, y_center`). Compute `slope_lev` and `slope_center`.
3. Compute `delta_lev` and `delta_center`: the change of the slope with respect to `slope_base`.
4. Write `leverages(x)` (diagonal of the hat matrix of a regression with intercept) and save `lev_A` and `lev_B`: the leverage of the added point in the two datasets.
5. On the dataset with the point at the edge compute `slope_huber` (`HuberRegressor`) and `slope_ransac` (`RANSACRegressor(random_state=0)`; the slope is in `model.estimator_.coef_`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import HuberRegressor, LinearRegression, RANSACRegressor

rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 50)
y = 2 + 1.5 * x + rng.normal(size=50)

## Your work

In [ ]:
def slope(x, y, model=None):
    ...

slope_base = slope(x, y)
slope_base

In [ ]:
x_lev, y_lev = ..., ...
x_center, y_center = ..., ...
slope_lev = ...
slope_center = ...
delta_lev = ...
delta_center = ...
print(f"slope: base {slope_base:.3f} | point at the edge {slope_lev:.3f} | point at the center {slope_center:.3f}")

In [ ]:
def leverages(x):
    ...

lev_A = ...
lev_B = ...

In [ ]:
slope_huber = ...
slope_ransac = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='To add a point: `np.append(x, 10.0)`. For the leverage use `A = np.column_stack([np.ones_like(x), x])` and `np.diag(A @ np.linalg.inv(A.T @ A) @ A.T)`; the added point is the last element (`[-1]`). With RANSAC the fitted model is in `model.estimator_`, not `model.coef_`.'):
    check(close(slope_base, np.polyfit(x, y, 1)[0]), "correct slope on the clean data",
          "slope_base should match the least squares slope")
    check(len(x_lev) == 51 and len(x_center) == 51, "you added a single point to each dataset",
          "x_lev and x_center must have 51 elements")
    check(abs(delta_lev) > 5 * abs(delta_center),
          "the same outlier moves the line MUCH more at the edges than in the center",
          "delta_lev should be at least 5 times delta_center: did you add y = -20 at x = 10 and at x = 5?")
    check(lev_A > 2 * lev_B, "the leverage of the point at the edge is much larger",
          "lev_A should be more than double lev_B")
    check(abs(sum(leverages(x)) - 2) < 1e-8, "the sum of the leverages is p = 2 (intercept + slope)",
          "the sum of the diagonal elements of the hat matrix must equal the number of parameters")
    check(abs(slope_huber - slope_base) < abs(delta_lev) / 3, "Huber recovers most of the true slope",
          "the Huber slope should stay much closer to slope_base than the OLS one")
    check(abs(slope_ransac - slope_base) < 0.1, "RANSAC ignores the outlier completely",
          "RANSAC should estimate a slope very close to the clean one")
done()

---
The commented solution is in `solutions/C_break_the_model/C2_outliers_and_leverage.ipynb`. Open it after you pass the check, or after you have really tried.